In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### phase 1

In [ ]:
df = pd.read_csv("employee_turnover.csv")

In [ ]:
df.describe()

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

### Phase 2 - EDA

In [ ]:
df['Employee_Turnover'].value_counts().plot(kind='bar')
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

In [ ]:
features_to_check = ['Job_Satisfaction', 'Years_At_Company', 'Num_Companies_Worked']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, feature in enumerate(features_to_check):
    df[df['Employee_Turnover'] == 0][feature].plot(
        kind='kde', ax=axes[i], label='Stayed', color='#1D9E75')
    df[df['Employee_Turnover'] == 1][feature].plot(
        kind='kde', ax=axes[i], label='Left', color='#E24B4A')
    axes[i].set_title(f'{feature} by Turnover')
    axes[i].legend()
plt.tight_layout()
plt.show()
# What to look for: do the two curves separate?
# If they overlap completely = feature is useless
# If they are shifted apart = feature is a good predictor

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()
features = [col for col in df.columns if col != 'Employee_Turnover']

for i, feature in enumerate(features):
    df.boxplot(column=feature, by='Employee_Turnover', ax=axes[i])
    axes[i].set_title(feature)
    axes[i].set_xlabel('0=Stayed, 1=Left')
plt.suptitle('Feature Distribution by Turnover')
plt.tight_layout()
plt.show()
# What to look for: is the median different between groups?
# Big median difference = strong predictor

In [ ]:
# ── 4. Correlation heatmap ───────────────────────────────
fig, ax = plt.subplots(figsize=(12, 8))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
df = df.drop(columns = ['Annual_Bonus_Squared'])

## EDA Summary

### Strong Predictors
`Job_Satisfaction`, `Performance_Rating`, `Years_At_Company`, 
`Work_Life_Balance`, and `Distance_From_Home` showed the strongest 
signal for predicting employee turnover. In the box plots, these 
features had noticeably different medians between employees who 
left vs employees who stayed — meaning the model can use these 
features to separate the two groups.

### Surprising Finding
`Num_Companies_Worked` was expected to be a strong predictor based 
on the job-hopper hypothesis — someone who has switched multiple 
companies before is more likely to switch again. However, the KDE 
plot showed significant overlap between leavers and stayers for 
this feature, suggesting it is a weaker predictor than expected 
in this dataset. This is a good reminder that domain intuition 
is a starting point, not a guarantee.

### Feature Removed
`Annual_Bonus_Squared` was removed due to multicollinearity 
(Pearson r = 0.97 with `Annual_Bonus`). Since it is directly 
derived from `Annual_Bonus`, keeping both would give the model 
redundant information and produce unstable coefficients in 
Logistic Regression.

In [ ]:
df.head()

### Phase 3 - Featuring Engineering ## Phase 3 — Train/Test Split

- Split: 80% train (1080 samples), 20% test (270 samples)
- Stratified split used to preserve class balance
- Train turnover rate: 0.498 | Test turnover rate: 0.496
- No additional scaling applied — data is already MinMax scaled (0–1)
- Features: 14 | Target: Employee_Turnover

before we do anything we do train/test split to prevent data leakage -  
If you scale or engineer features on the full dataset first, then split — your test set has already influenced your scaler. The model has indirectly "seen" the test data. Your evaluation becomes fake.

since our data is already minmax scaled there is no need to standardize it 

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x = df.drop(columns = 'Employee_Turnover')
y = df['Employee_Turnover']
print(x.shape)
print(y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x,y,test_size = 0.2,stratify=y,random_state = 42)
print(X_train.shape) 
print(X_test.shape)

If your classes are imbalanced, a random split might accidentally put too many majority-class samples in train and too few minority-class samples in test, or even miss the minority class in one split. Stratifying avoids that and makes the split more representative of the real dataset distribution.

In [ ]:
print(f"Train turnover rate: {y_train.mean():.3f}")
print(f"Test turnover rate:  {y_test.mean():.3f}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score,recall_score, f1_score,roc_auc_score, classification_report,
                            ConfusionMatrixDisplay)

In [ ]:
def evaluate_model(name,model,X_test,)